# 🛡️ SafeNest Section 3: Thermal-44 Camera (80x62 IR Array) Fall Detection AI
### Google Colab 전용 Keras 모델 훈련 & INT8 Full Quantization TFLite 변환 파이프라인

본 노트북은 **Google Drive (`MyDrive/SafeNest/processed_thermal_80x62.npz`)** 경로의 **Thermal-44 Camera 80x62 초저해상도 IR Array 센서 (I2C+SPI 인터페이스, 총 4,960 픽셀)** 체온 배열 데이터를 바탕으로 바닥 쓰러짐/기절 자세(Posture)를 실시간 감지하는 경량 신경망을 훈련하고 INT8 양자화 TFLite 모델을 추출합니다.
- **센서 모듈 스펙**: Thermal-44 Camera (80x62, I2C + SPI)
- **구글 드라이브 경로**: `MyDrive/SafeNest/processed_thermal_80x62.npz`
- **입력 텐서 규격**: `(62, 80, 1)`
- **최적 학습 셋팅**: `Epochs = 100 (대량 에폭 설정)`, `EarlyStopping(patience=10)`, `ReduceLROnPlateau` 적용

## 1. 구글 드라이브 마운트 및 SafeNest 폴더 내 데이터셋 로드 (`MyDrive/SafeNest/processed_thermal_80x62.npz`)

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np

# 1. 구글 드라이브 마운트 실행
drive_mount_path = Path.cwd() / 'drive'
drive.mount(str(drive_mount_path))

# 2. 구글 드라이브 SafeNest 폴더 내 80x62 데이터셋 경로 설정
dataset_path = drive_mount_path / 'MyDrive' / 'SafeNest' / 'processed_thermal_80x62.npz'

if not dataset_path.is_file():
    raise FileNotFoundError(
        f"❌ 구글 드라이브 경로에 파일이 존재하지 않습니다: {dataset_path}\n"\
        "👉 내 구글 드라이브(MyDrive)에 'SafeNest' 폴더를 만들고 processed_thermal_80x62.npz 파일을 넣어주세요!"
    )

data = np.load(dataset_path)
X_raw, y = data['X'], data['y']
X = np.expand_dims(X_raw, axis=-1) # (N, 62, 80, 1)

print(f"✅ SafeNest 구글 드라이브 데이터셋 로드 완료: Shape={X.shape}, 총 프레임 수={len(y)}")
print(f"   - 클래스 분포: {dict(zip(*np.unique(y, return_counts=True)))}")

## 2. Keras 경량 2D-CNN (Thermal-44 80x62) 모델 설계 및 대량 100 에폭 최적 훈련

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# 80:20 Train/Test 분할
indices = np.random.permutation(len(X))
split_idx = int(len(X) * 0.8)
X_train, X_test = X[indices[:split_idx]], X[indices[split_idx:]]
y_train, y_test = y[indices[:split_idx]], y[indices[split_idx:]]

# Thermal-44 80x62 맞춤 2D-CNN 경량 아키텍처
def create_thermal_fall_model():
    model = models.Sequential([
        layers.Input(shape=(62, 80, 1)),
        layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.DepthwiseConv2D((3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (1, 1), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.25),
        layers.Dense(16, activation='relu'),
        layers.Dense(3, activation='softmax') # [0: Not Human, 1: Normal, 2: Fall]
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = create_thermal_fall_model()
model.summary()

# 대량 100 에폭 설정 및 과적합 방지 최적 가중치 복원 콜백 (patience=10)
cb_early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
)
cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=4, verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100, # 넉넉하게 100 에폭으로 확장!
    batch_size=32,
    callbacks=[cb_early_stop, cb_reduce_lr]
)

## 3. INT8 Representative Calibration TFLite 양자화 변환 및 구글드라이브 자동 저장

In [ ]:
from google.colab import files

# Representative Dataset Calibration 제너레이터
def representative_dataset_gen():
    for i in range(min(300, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_quant = converter.convert()

# 구글 드라이브 SafeNest 폴더로 바로 저장
drive_save_path = drive_mount_path / 'MyDrive' / 'SafeNest' / 'thermal_fall_quant.tflite'
with open(drive_save_path, 'wb') as f:
    f.write(tflite_model_quant)

print(f'🎉 INT8 TFLite 모델 변환 완료 및 구글 드라이브 저장 성공!')
print(f'   - 드라이브 저장 경로: {drive_save_path}')
print(f'   - 파일 용량: {len(tflite_model_quant)/1024:.2f} KB')

# 로컬 자동 다운로드 실행
files.download(str(drive_save_path))